# Semanas 0 y 1: entorno, git y contexto físico

Programa: [`docs/programa_mes1.md`](../../docs/programa_mes1.md), secciones *Semana 0* y *Semana 1*.

Esta semana casi todo pasa fuera del notebook (GitHub, terminal, lecturas). Aquí están las verificaciones que sí
se pueden comprobar con código (E0.1, E0.2, E0.3, E0.5, E1.3, E1.5) y el espacio para tus respuestas.

**Cómo trabajar este notebook.** Completa las celdas marcadas con `# TODO`. Las celdas de **verificación**
(las que terminan en `assert`) ya están escritas: no las borres ni cambies las tolerancias. Si una verificación
falla, intenta corregirlo tú; si sigue fallando, anótalo en `bitacora.md` en *qué no entendí*. Las celdas
*Tu respuesta* se contestan con tus palabras (2 a 5 frases) y se discuten en la reunión.

Antes del commit: *Kernel → Restart & Run All* (todo debe correr de principio a fin) y luego borra las salidas.

In [ ]:
# --- Configuración: funciona igual en Google Colab y en un entorno conda local ---
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/RREYESAL/Hands_on_SSIPN"  # <- si trabajas en un fork, pon aquí su URL
DIR_COLAB = Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "uproot", "awkward", "vector", "hist", "mplhep", "pyyaml", "aiohttp", "requests", "psutil"], check=True)
    if not DIR_COLAB.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(DIR_COLAB)], check=True)
    os.chdir(DIR_COLAB / "notebooks" / "mes1")

# Buscar la raíz del repositorio (la carpeta que contiene src/hepss) y hacer importable hepss
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hepss").is_dir()), None)
if RAIZ is None:
    raise RuntimeError("No encuentro src/hepss: corre el notebook desde dentro del repositorio")
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))
os.chdir(RAIZ)  # rutas relativas (configs/, data/, plots/) respecto a la raíz
Path("plots").mkdir(exist_ok=True)

# Recarga automática de hepss: si editas src/hepss/*.py, no hace falta reiniciar el kernel
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

# Para forzar los datos sintéticos (sin red), descomentar:
# os.environ["HEPSS_SINTETICO"] = "1"
print("Colab:", EN_COLAB, "| raíz del repositorio:", RAIZ)

## E0.1 Repositorio

Clona el repositorio (ruta A) o ábrelo desde Colab (ruta B). Si trabajas en un fork, cambia `REPO_URL` en la celda de
arriba. Revisa que tengas la estructura del programa. Ojo: git no guarda carpetas vacías, así que `data/` y `plots/`
no vienen en el repositorio; la celda de configuración crea `plots/` y `hepss` crea `data/` cuando hace falta.

In [ ]:
for carpeta in ["notebooks/mes1", "src/hepss", "configs", "docs"]:
    assert (RAIZ / carpeta).is_dir(), f"Falta la carpeta {carpeta}"
for archivo in ["README.md", "environment.yml", "requirements.txt", "bitacora.md", ".gitignore"]:
    assert (RAIZ / archivo).exists(), f"Falta {archivo}"
ignorado = (RAIZ / ".gitignore").read_text()
assert "*.root" in ignorado and "data/" in ignorado, ".gitignore debe excluir datos y archivos ROOT"
print("Estructura correcta")

## E0.2 Entorno

Ruta A: `conda env create -f environment.yml` y `conda activate hep-ss`. Ruta B: la primera celda instala lo necesario
para el mes 1. Esta celda comprueba que todo se importe; las librerías de los meses 2 y 3 se comprueban ya para que los
problemas de instalación aparezcan ahora y no en la semana 5. En Colab, `zfit` instala TensorFlow; si Colab pide reiniciar
la sesión después del `pip install`, se reinicia y se vuelve a correr desde la primera celda.

In [ ]:
import importlib

nucleo = ["numpy", "pandas", "matplotlib", "uproot", "awkward", "vector", "hist", "mplhep", "yaml"]
posteriores = ["scipy", "iminuit", "zfit", "sklearn", "xgboost"]   # meses 2 y 3

def version(nombre):
    try:
        m = importlib.import_module(nombre)
        return getattr(m, "__version__", "ok")
    except Exception as e:
        return f"FALTA ({type(e).__name__})"

for n in nucleo + posteriores:
    print(f"{n:12s} {version(n)}")

faltan_nucleo = [n for n in nucleo if version(n).startswith("FALTA")]
assert not faltan_nucleo, f"Faltan librerías del mes 1: {faltan_nucleo}"
faltan_despues = [n for n in posteriores if version(n).startswith("FALTA")]
if faltan_despues:
    print("\nAviso: faltan librerías de los meses 2-3 (no bloquean el mes 1):", faltan_despues)

## E0.3 Paquete `hepss`

Ruta A: `pip install -e .` en la raíz (ya lo hace `environment.yml`). Ruta B: la celda de configuración añade `src/`
a `sys.path`. La verificación pide una sesión nueva: reinicia el kernel y corre solo la celda de configuración y esta.

In [ ]:
import hepss
print("hepss", hepss.__version__, "desde", Path(hepss.__file__).parent)
assert Path(hepss.__file__).parent.resolve() == (RAIZ / "src" / "hepss").resolve()

## E0.4 Notebooks limpios

Ruta A: instala el filtro `nbstripout` en el repositorio, haz un commit y push desde la terminal y comprueba en GitHub
que el notebook subido no tiene salidas. Ruta B: *Editar → Borrar todas las salidas* y luego *Archivo → Guardar una copia
en GitHub* en una rama nueva. Busca en la documentación de `nbstripout` cómo instalarlo como filtro de git.

¿Cómo reconoces en la pestaña *Files changed* de un PR que un notebook tiene salidas?

**Tu respuesta:** _(escribe aquí)_

## E0.5 Guardar resultados intermedios

Leer datos es lo lento; histogramar es rápido. Por eso se guarda el histograma, no los eventos.
Crea un histograma `h` de `hist` (con `storage=hist.storage.Weight()`) llenado con 10 000 números aleatorios,
guárdalo en `DIR_DATOS` como `ejemplo_hist.pkl` (con `pickle`) y como `ejemplo_hist.npz` (con `np.savez`: contenidos,
varianzas y bordes), y vuelve a cargarlos en `h2` y `d`.

En Colab, `data/` se borra al cerrar la sesión: monta Drive antes
(`from google.colab import drive; drive.mount('/content/drive')`).

In [ ]:
import pickle
import numpy as np
import hist

DIR_DATOS = Path("/content/drive/MyDrive/hep-ss-data") if EN_COLAB and Path("/content/drive").exists() else RAIZ / "data"
DIR_DATOS.mkdir(parents=True, exist_ok=True)

# TODO: crea y llena h, guárdalo en .pkl y en .npz, y vuelve a cargarlos
h = ...
h2 = ...
d = ...

In [ ]:
# Verificación E0.5
assert h2 == h
assert np.array_equal(d["valores"], h.values()) and np.array_equal(d["bordes"], h.axes[0].edges)
print("Histograma recuperado sin volver a generar los datos:", h2.sum())

## E0.6, E1.1 y E1.2 Bitácora y git

* E0.6: primera entrada de `bitacora.md` con la plantilla que viene en el archivo.
* E1.1: rama `semana-01-contexto`, cambio al README (ruta elegida y cómo instalar) y un PR con descripción.
* E1.2: en una rama de prueba, provoca a propósito un conflicto de merge (dos ramas que cambian la misma línea de un
  archivo) y resuélvelo. Escribe en la bitácora los comandos que usaste y qué significan las dos mitades entre los
  marcadores `<<<<<<<`, `=======` y `>>>>>>>`.

Referencia: módulo *Version control with git* de [HSF Training](https://hsf-training.org).

## E1.3 `docs/contexto.md`

Escribe en `docs/contexto.md` (máximo una página): qué es un mesón Bs, por qué se estudian Bs → J/ψ φ y Bs → μ⁺μ⁻,
y por qué el J/ψ se reconstruye con dos muones. Debe contener la masa y la vida media del Bs con su fuente (PDG).

In [ ]:
# Verificación E1.3 (solo comprueba que exista y cite su fuente; el contenido se revisa en el PR)
texto = (RAIZ / "docs" / "contexto.md").read_text()
assert "PDG" in texto, "Cita la fuente (PDG) de la masa y la vida media"
assert len(texto.split()) < 700, "Máximo una página"
print(len(texto.split()), "palabras")

## E1.4 Cadena detector → notebook

Completa la tabla (puedes copiarla a `docs/contexto.md`). Cada paso lleva una frase sobre qué información se pierde
o se gana.

| Paso | Qué hace | Qué se pierde o se gana |
| --- | --- | --- |
| Detector | | |
| Trigger (L1 y HLT) | | |
| Reconstrucción offline | | |
| NanoAOD | | |
| Notebook | | |

## E1.5 Distancia de vuelo de un Bs

Calcula la distancia media que recorre un Bs con pT = 10 GeV antes de decaer, usando cτ del PDG y βγ = p/m.
Escríbelo como función `distancia_media_mm(pt, eta=0.0)` que devuelva milímetros. Pon las constantes del PDG en
`src/hepss/constantes.py` (hay un `TODO` ahí) con su fuente. Cuidado con las unidades.

In [ ]:
import numpy as np

# TODO: completa las constantes en src/hepss/constantes.py e impórtalas aquí
# from hepss.constantes import M_BS, TAU_BS_PS

def distancia_media_mm(pt, eta=0.0):
    # TODO
    ...

L0 = distancia_media_mm(10.0)
print(f"<L>(pT = 10 GeV, η = 0) = {L0:.2f} mm")

In [ ]:
# Verificación E1.5
assert 0.5 < L0 < 1.5, "Debe ser del orden de 1 mm"
assert distancia_media_mm(10.0, eta=1.0) > L0, "Con η ≠ 0 el momento es mayor que pT"

¿Por qué una distancia de vuelo de ~1 mm permite separar la señal de Bs del fondo? ¿Qué resolución hace falta para aprovecharlo?

**Tu respuesta:** _(escribe aquí)_